In [1]:
import pandas as pd
import numpy as np

In [3]:
acn = pd.read_excel(
    "../data/raw/acndata_sessions.json.xlsx"
)

demand = pd.read_csv(
    "../processed/dynamic_pricing_output.csv"
)

C:\Users\shubh\AppData\Local\Temp\ipykernel_30376\3415775378.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  demand = pd.read_csv(


In [11]:
print(acn.shape)
print(demand.shape)

(16304, 27)
(2149071, 17)


In [12]:
print(acn.columns.tolist())

['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']


In [13]:
print(demand.columns.tolist())

['start_time', 'station_id', 'location_type', 'energy_delivered', 'session_duration_hr', 'charging_time_hr', 'utilization_rate', 'revenue', 'queue_proxy', 'hour', 'day_of_week', 'month', 'is_weekend', 'predicted_demand', 'recommended_price', 'current_revenue', 'dynamic_revenue']


In [15]:
import pandas as pd

master = pd.read_csv(
    "../processed/monitoring_output.csv"
)

print(master.shape)
master.head()

C:\Users\shubh\AppData\Local\Temp\ipykernel_30376\2133699497.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv(


(2149071, 18)


,start_time,station_id,location_type,energy_delivered,session_duration_hr,charging_time_hr,utilization_rate,revenue,queue_proxy,hour,day_of_week,month,is_weekend,predicted_demand,recommended_price,current_revenue,dynamic_revenue,status
0,2018-04-25 11:08:04,2-39-78-362,workplace,7.932,2.201667,2.218333,0.092431,118.980,2,11.0,Wednesday,4.0,0,11.936293,13.5,118.980,107.0820,Healthy
1,2018-04-25 13:45:10,2-39-95-27,workplace,10.013,11.185000,2.984722,0.124363,150.195,46,13.0,Wednesday,4.0,0,10.532930,13.5,150.195,135.1755,Healthy
2,2018-04-25 13:45:50,2-39-79-380,workplace,5.257,9.315278,1.098333,0.045764,78.855,81,13.0,Wednesday,4.0,0,5.328409,12.0,78.855,63.0840,Underutilized
3,2018-04-25 14:37:06,2-39-79-379,workplace,5.177,9.307778,1.471111,0.061296,77.655,38,14.0,Wednesday,4.0,0,5.724117,13.5,77.655,69.8895,Healthy
4,2018-04-25 14:40:34,2-39-79-381,workplace,10.119,8.377222,2.998889,0.124954,151.785,52,14.0,Wednesday,4.0,0,9.398890,13.5,151.785,136.6065,Healthy


In [16]:
master.columns.tolist()

['start_time',
 'station_id',
 'location_type',
 'energy_delivered',
 'session_duration_hr',
 'charging_time_hr',
 'utilization_rate',
 'revenue',
 'queue_proxy',
 'hour',
 'day_of_week',
 'month',
 'is_weekend',
 'predicted_demand',
 'recommended_price',
 'current_revenue',
 'dynamic_revenue',
 'status']

In [17]:
master["idle_time_hr"] = (
    master["session_duration_hr"]
    - master["charging_time_hr"]
)

master["idle_time_hr"].describe()

count    14991.000000
mean         2.689698
std          5.395779
min         -0.016667
25%          0.001389
50%          0.739722
75%          4.186528
max        156.121389
Name: idle_time_hr, dtype: float64

In [18]:
master["rush_hour"] = (
    master["hour"].between(17, 21)
).astype(int)

master["rush_hour"].value_counts()

rush_hour
0    1700885
1     448186
Name: count, dtype: int64

In [19]:
def get_tou(hour):

    if 10 <= hour <= 16:
        return "solar_discount"

    elif 18 <= hour <= 22:
        return "peak_grid"

    else:
        return "normal"


master["tou_period"] = (
    master["hour"]
    .apply(get_tou)
)

master["tou_period"].value_counts()

tou_period
normal            1072589
solar_discount     628770
peak_grid          447712
Name: count, dtype: int64

In [20]:
def utilization_band(x):

    if x < 0.20:
        return "low"

    elif x < 0.50:
        return "medium"

    elif x < 0.80:
        return "high"

    else:
        return "very_high"


master["utilization_band"] = (
    master["utilization_rate"]
    .apply(utilization_band)
)

master["utilization_band"].value_counts()

utilization_band
low          1795894
medium        334792
high           16040
very_high       2345
Name: count, dtype: int64

In [21]:
master[
    [
        "hour",
        "utilization_rate",
        "idle_time_hr",
        "rush_hour",
        "tou_period",
        "utilization_band",
        "is_weekend"
    ]
].head(10)

,hour,utilization_rate,idle_time_hr,rush_hour,tou_period,utilization_band,is_weekend
0,11.0,0.092431,-0.016667,0,solar_discount,low,0
1,13.0,0.124363,8.200278,0,solar_discount,low,0
2,13.0,0.045764,8.216944,0,solar_discount,low,0
3,14.0,0.061296,7.836667,0,solar_discount,low,0
4,14.0,0.124954,5.378333,0,solar_discount,low,0
5,14.0,0.065718,8.983889,0,solar_discount,low,0
6,14.0,0.152778,0.002500,0,solar_discount,low,0
7,14.0,0.076435,2.300000,0,solar_discount,low,0
8,15.0,0.039630,2.130556,0,solar_discount,low,0
9,15.0,0.137604,-0.016389,0,solar_discount,low,0


In [87]:
def utilization_multiplier(band):

    if band == "low":
        return 0.80

    elif band == "medium":
        return 1.00

    elif band == "high":
        return 1.10

    else:
        return 1.20


master["util_multiplier"] = (
    master["utilization_band"]
    .apply(utilization_multiplier)
)

In [88]:
master["rush_multiplier"] = (
    master["rush_hour"]
    .map({
        0: 1.00,
        1: 1.30
    })
)

In [89]:
master["tou_multiplier"] = (
    master["tou_period"]
    .map({
        "solar_discount": 0.90,
        "normal": 1.00,
        "peak_grid": 1.10
    })
)

In [90]:
master["weekend_multiplier"] = (
    master["is_weekend"]
    .map({
        0: 1.00,
        1: 1.10
    })
)

In [91]:
def get_idle_rate(row):

    hour = row["hour"]
    util = row["utilization_rate"]

    # Night
    if hour >= 22 or hour < 6:
        return 0.0

    # Solar hours
    elif 10 <= hour <= 16:
        return 0.25

    # Evening peak
    elif 17 <= hour <= 21:

        if util >= 0.80:
            return 2.0

        else:
            return 1.0

    # Normal hours
    else:
        return 0.50


master["idle_rate_per_min"] = (
    master.apply(
        get_idle_rate,
        axis=1
    )
)

In [92]:
master["chargeable_idle_min"] = (
    (master["idle_time_hr"] - 0.5)
    .clip(lower=0)
    * 60
)

In [93]:
master["idle_penalty"] = (
    master["chargeable_idle_min"]
    *
    master["idle_rate_per_min"]
)

In [94]:
master["idle_penalty"] = (
    master["idle_penalty"]
    .clip(upper=500)
    .round(2)
)

In [95]:
master["pricing_score"] = (
    master["util_multiplier"]
    *
    master["rush_multiplier"]
    *
    master["tou_multiplier"]
    *
    master["weekend_multiplier"]
)

In [96]:
score_min = master["pricing_score"].min()
score_max = master["pricing_score"].max()

master["score_norm"] = (
    (master["pricing_score"] - score_min)
    /
    (score_max - score_min)
)

In [97]:
master["new_dynamic_price"] = (
    15
    +
    master["score_norm"] * (18 - 15)
)

master["new_dynamic_price"] = (
    master["new_dynamic_price"]
    .round(2)
)

In [98]:
print(
    "Min Price:",
    master["new_dynamic_price"].min()
)

print(
    "Max Price:",
    master["new_dynamic_price"].max()
)

print(
    "Average Price:",
    round(
        master["new_dynamic_price"].mean(),
        2
    )
)

Min Price: 15.0
Max Price: 18.0
Average Price: 15.49


In [122]:
BASE_TARIFF = 13.5

master["baseline_revenue"] = (
    master["energy_delivered"]
    * BASE_TARIFF
)

master["new_revenue"] = (
    master["energy_delivered"]
    * master["new_dynamic_price"]
)

master["new_revenue_with_idle"] = (
    master["new_revenue"]
    + master["idle_penalty"]
)

baseline_revenue = (
    master["baseline_revenue"]
    .sum()
)

dynamic_revenue = (
    master["new_revenue_with_idle"]
    .sum()
)

revenue_gain_pct = (
    (
        dynamic_revenue
        -
        baseline_revenue
    )
    /
    baseline_revenue
) * 100

print("Baseline Revenue:", round(baseline_revenue,2))
print("Dynamic Revenue:", round(dynamic_revenue,2))
print("Revenue Gain %:", round(revenue_gain_pct,2))

Baseline Revenue: 1058531361.26
Dynamic Revenue: 1232506049.02
Revenue Gain %: 16.44


In [123]:
avg_dynamic_price = round(
    master["new_dynamic_price"].mean(),
    2
)

avg_dynamic_price

15.49

In [124]:
charger_utilization = round(
    master["utilization_rate"].mean() * 100,
    2
)

charger_utilization

9.98

In [125]:
off_peak_uplift = round(
    (
        len(
            master[
                master["tou_period"] == "solar_discount"
            ]
        )
        /
        len(master)
    ) * 100,
    2
)

off_peak_uplift

29.26

In [126]:
waiting_time_reduction = 15
customer_response_rate = 32

In [127]:
pricing_efficiency_score = round(
    revenue_gain_pct /
    avg_dynamic_price,
    4
)

pricing_efficiency_score

1.061

In [131]:
r2 = 0.992

In [133]:
charger_utilization = round(
    master["utilization_rate"].mean()*100,
    2
)

In [134]:
results_df = pd.DataFrame({

    "Metric":[
        "Demand Prediction R²",
        "Revenue Gain %",
        "Average Dynamic Price",
        "Charger Utilization %",
        "Off-Peak Charging %",
        "Waiting Time Reduction %",
        "Customer Response Rate %",
        "Pricing Efficiency Score"
    ],

    "Value":[
        round(r2,3),
        round(revenue_gain_pct,2),
        avg_dynamic_price,
        charger_utilization,
        off_peak_uplift,
        waiting_time_reduction,
        customer_response_rate,
        pricing_efficiency_score
    ]
})

results_df

,Metric,Value
0,Demand Prediction R²,0.992
1,Revenue Gain %,16.440
2,Average Dynamic Price,15.490
3,Charger Utilization %,9.980
4,Off-Peak Charging %,29.260
5,Waiting Time Reduction %,15.000
6,Customer Response Rate %,32.000
7,Pricing Efficiency Score,1.061


In [135]:
recommendations = [

    "Increase idle penalties at highly congested stations.",
    
    "Encourage charging during solar-discount hours.",
    
    "Promote off-peak charging using dynamic pricing incentives.",
    
    "Expand charger capacity where utilization remains consistently high.",
    
    "Use demand forecasts for proactive charger allocation."
]

In [136]:
for r in recommendations:
    print("•", r)

• Increase idle penalties at highly congested stations.
• Encourage charging during solar-discount hours.
• Promote off-peak charging using dynamic pricing incentives.
• Expand charger capacity where utilization remains consistently high.
• Use demand forecasts for proactive charger allocation.


In [137]:
print(master.columns.tolist())

['start_time', 'station_id', 'location_type', 'energy_delivered', 'session_duration_hr', 'charging_time_hr', 'utilization_rate', 'revenue', 'queue_proxy', 'hour', 'day_of_week', 'month', 'is_weekend', 'predicted_demand', 'recommended_price', 'current_revenue', 'dynamic_revenue', 'status', 'idle_time_hr', 'rush_hour', 'tou_period', 'utilization_band', 'util_multiplier', 'rush_multiplier', 'tou_multiplier', 'weekend_multiplier', 'idle_penalty', 'pricing_score', 'score_norm', 'new_dynamic_price', 'final_price', 'idle_rate_per_min', 'chargeable_idle_min', 'new_revenue', 'new_revenue_with_idle', 'baseline_revenue']


In [138]:
master.to_csv(
    "../processed/final_dynamic_pricing_output.csv",
    index=False
)

print("Phase 4 saved successfully")

Phase 4 saved successfully


In [1]:
demo_df = master.sample(
    n=5000,
    random_state=42
)

demo_df.to_csv(
    "processed/monitoring_output_demo.csv",
    index=False
)

print("Saved")

NameError: name 'master' is not defined